# 01 — Data Exploration & EDA

This notebook performs a thorough exploratory data analysis of the raw e-commerce dataset.

**Objectives:**
- Understand dataset structure and distributions
- Identify missing values and data quality issues
- Discover patterns in sales, customers, and products

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

from src.data_loader  import DataLoader
from src.data_cleaner import DataCleaner

# ── Style ──
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5),
                     'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold'})

VISUALS = Path('../visuals')
VISUALS.mkdir(exist_ok=True)
print("Libraries loaded.")

## 1. Load & Clean Data

In [ ]:
loader  = DataLoader()
df_raw  = loader.load_raw_data()
cleaner = DataCleaner()
df      = cleaner.clean(df_raw)
print(f"Dataset shape: {df.shape}")
df.head(3)

## 2. Dataset Overview

In [ ]:
print(f"Rows           : {len(df):,}")
print(f"Columns        : {df.shape[1]}")
print(f"Date range     : {df['order_date'].min().date()} to {df['order_date'].max().date()}")
print(f"Unique customers: {df['customer_id'].nunique():,}")
print(f"Unique products : {df['product_id'].nunique():,}")
print(f"Unique cities   : {df['customer_city'].nunique()}")
print(f"\nOrder Status Distribution:")
print(df['order_status'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
# Missing values heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

null_pct = df.isnull().mean().mul(100).sort_values(ascending=False)
null_pct[null_pct > 0].plot(kind='bar', ax=axes[0], color='#ef4444', edgecolor='white')
axes[0].set_title('Missing Values (%)')
axes[0].set_ylabel('% Missing')
axes[0].tick_params(axis='x', rotation=45)

df.dtypes.value_counts().plot(kind='pie', ax=axes[1], autopct='%1.0f%%',
                               colors=['#00d4ff','#7c3aed','#22c55e'])
axes[1].set_title('Data Types Distribution')
axes[1].set_ylabel('')
plt.tight_layout()
plt.savefig(VISUALS / 'data_overview.png', bbox_inches='tight')
plt.show()
print("Saved: data_overview.png")

## 3. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Age distribution
axes[0].hist(df['customer_age'], bins=20, color='#00d4ff', edgecolor='white', alpha=0.8)
axes[0].set_title('Customer Age Distribution')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Count')

# Unit price distribution
axes[1].hist(df[df['unit_price'] < 50000]['unit_price'], bins=40,
             color='#7c3aed', edgecolor='white', alpha=0.8)
axes[1].set_title('Unit Price Distribution (< 50k)')
axes[1].set_xlabel('Price (INR)')

# Rating distribution
df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[2],
    color=['#ef4444','#f59e0b','#eab308','#22c55e','#06b6d4'], edgecolor='white')
axes[2].set_title('Rating Distribution')
axes[2].set_xlabel('Rating (1-5)')

# Gender breakdown
df['customer_gender'].value_counts().plot(kind='pie', ax=axes[3], autopct='%1.1f%%',
    colors=['#00d4ff','#7c3aed','#f59e0b'], startangle=90)
axes[3].set_title('Gender Distribution'); axes[3].set_ylabel('')

# Device type
df['device_type'].value_counts().plot(kind='bar', ax=axes[4],
    color=['#22c55e','#f59e0b','#ef4444'], edgecolor='white')
axes[4].set_title('Device Type Distribution')
axes[4].set_xlabel('Device')

# Delivery days
axes[5].hist(df['delivery_days'], bins=15, color='#10b981', edgecolor='white', alpha=0.8)
axes[5].set_title('Delivery Days Distribution')
axes[5].set_xlabel('Days')

plt.suptitle('Univariate Analysis — Key Variables', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(VISUALS / 'univariate_analysis.png', bbox_inches='tight')
plt.show()

## 4. Category & Revenue Analysis

In [ ]:
completed = df[df['order_status'] == 'Completed']

# Revenue by category
cat_rev = completed.groupby('category')['total_amount'].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cat_rev.plot(kind='barh', ax=axes[0], color='#00d4ff', edgecolor='white')
axes[0].set_title('Revenue by Category')
axes[0].set_xlabel('Total Revenue (INR)')

# Orders by category
cat_orders = df.groupby('category')['order_id'].nunique().sort_values(ascending=False)
cat_orders.plot(kind='bar', ax=axes[1], color='#7c3aed', edgecolor='white')
axes[1].set_title('Number of Orders by Category')
axes[1].set_xlabel('Category')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Category Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VISUALS / 'category_sales.png', bbox_inches='tight')
plt.show()
print("Saved: category_sales.png")

## 5. Time-based Analysis — Monthly Revenue Trend

In [ ]:
monthly = (completed.groupby('order_month_year')['total_amount']
           .sum().reset_index())
monthly.columns = ['month', 'revenue']
monthly['month'] = pd.to_datetime(monthly['month'])
monthly = monthly.sort_values('month')

monthly_orders = (df.groupby('order_month_year')['order_id']
                  .nunique().reset_index())
monthly_orders.columns = ['month', 'orders']
monthly_orders['month'] = pd.to_datetime(monthly_orders['month'])
monthly_orders = monthly_orders.sort_values('month')

fig, ax1 = plt.subplots(figsize=(16, 6))
ax2 = ax1.twinx()

ax1.fill_between(monthly['month'], monthly['revenue'], alpha=0.3, color='#00d4ff')
ax1.plot(monthly['month'], monthly['revenue'], color='#00d4ff', linewidth=2.5, marker='o', markersize=4)
ax2.plot(monthly_orders['month'], monthly_orders['orders'],
         color='#f59e0b', linewidth=2, linestyle='--', marker='s', markersize=4)

ax1.set_ylabel('Revenue (INR)', color='#00d4ff', fontsize=12)
ax2.set_ylabel('Number of Orders', color='#f59e0b', fontsize=12)
ax1.set_xlabel('Month')
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))
plt.title('Monthly Revenue & Orders Trend (2022–2024)', fontsize=14, fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], color='#00d4ff', lw=2, label='Revenue'),
    Line2D([0],[0], color='#f59e0b', lw=2, linestyle='--', label='Orders'),
]
ax1.legend(handles=legend_elements, loc='upper left')
plt.tight_layout()
plt.savefig(VISUALS / 'monthly_revenue_trend.png', bbox_inches='tight')
plt.show()
print("Saved: monthly_revenue_trend.png")

## 6. Geographic Analysis

In [ ]:
state_rev = (completed.groupby('customer_state')['total_amount']
             .sum().sort_values(ascending=False).head(10))
city_orders = (df.groupby('customer_city')['order_id']
               .nunique().sort_values(ascending=False).head(10))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
state_rev.plot(kind='bar', ax=axes[0], color='#06b6d4', edgecolor='white')
axes[0].set_title('Top 10 States by Revenue')
axes[0].set_xlabel('State')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))
axes[0].tick_params(axis='x', rotation=45)

city_orders.plot(kind='barh', ax=axes[1], color='#8b5cf6', edgecolor='white')
axes[1].set_title('Top 10 Cities by Orders')
axes[1].set_xlabel('Orders')

plt.suptitle('Geographic Distribution of Sales', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VISUALS / 'geographic_analysis.png', bbox_inches='tight')
plt.show()

## 7. Key Findings

| # | Finding |
|---|---|
| 1 | Q4 (Oct–Dec) consistently drives highest revenue due to festive season |
| 2 | Electronics is the top revenue category despite higher price points |
| 3 | Mobile devices account for ~60% of all orders, highlighting mobile-first strategy |
| 4 | UPI is the most popular payment method |
| 5 | Mumbai, Delhi, and Bangalore are the top 3 cities by order volume |
| 6 | Avg delivery time is 8-9 days; Fast delivery (<5 days) correlates with higher ratings |